# PV SCADA + Meteorological Dataset Builder — validated

Dataset sumber: Rostamipour, Alireza; Ghayeni, Amir Hossein (2026), “PV SCADA data”, Mendeley Data, V2, DOI: `10.17632/9fdvyjrf67.2`.

Tujuan notebook ini adalah membangun dataset terintegrasi pada resolusi SCADA 30 menit dengan mempertahankan variabel DC dan AC dari SCADA, serta mengagregasikan tiga weather station sebelum time alignment.

Perubahan utama:
- `dc_power_kw`, `ac_power_kw`, `dc_voltage_v`, `dc_current_a`, `module_temp_c`, `inverter_efficiency`, dan `status_code` dipertahankan.
- Nilai `dc_current_a` divalidasi terhadap hubungan fisik `P_DC = V_DC × I_DC / 1000`. Pada file sumber yang digunakan, angka current konsisten dengan skala kA meskipun nama kolom menyatakan A, sehingga nilai dikonversi ke A dengan faktor 1000.
- Tiga weather station diagregasi pada timestamp 15 menit menggunakan median, kemudian di-resample ke 30 menit.
- Alignment meteorologi ke SCADA dilakukan pada timestamp yang sama; tidak lagi memilih station secara arbitrer melalui `merge_asof`.
- Validasi unit dan physical consistency dilaporkan sebagai statistik/diagnostic. Tidak ada label fault/anomaly yang dibuat oleh notebook ini; anomaly detection tetap menjadi tanggung jawab model.


In [9]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("PV SCADA data")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SCADA_FILE = DATA_DIR / "Inverter SCADA.csv"
METEO_FILE = DATA_DIR / "Meteorological data.csv"
OUTPUT_FILE = OUTPUT_DIR / "pv_monitoring_dataset_validated.csv"
        


In [10]:
scada = pd.read_csv(SCADA_FILE)
meteo = pd.read_csv(METEO_FILE)

print("SCADA shape:", scada.shape)
print("Meteorological shape:", meteo.shape)
print("\nSCADA columns:", scada.columns.tolist())
print("Meteorological columns:", meteo.columns.tolist())


SCADA shape: (34560, 9)
Meteorological shape: (25920, 13)

SCADA columns: ['timestamp', 'inverter_id', 'dc_power_kw', 'ac_power_kw', 'dc_voltage_v', 'dc_current_a', 'module_temp_c', 'inverter_efficiency', 'status_code']
Meteorological columns: ['station_id', 'reading_time', 'ghi_wm2', 'dni_wm2', 'dhi_wm2', 'ambient_temp', 'temp_unit', 'wind_speed_ms', 'relative_humidity_pct', 'barometric_pressure_hpa', 'solar_zenith_angle_deg', 'cloud_opacity_pct', 'precipitation_mm']


## 1. Schema and timestamp validation

Validasi ini hanya memastikan struktur data sesuai sumber. Baris dengan timestamp invalid dibuang karena tidak dapat digunakan untuk time-series alignment.

In [11]:
SCADA_REQUIRED = [
    "timestamp", "inverter_id", "dc_power_kw", "ac_power_kw",
    "dc_voltage_v", "dc_current_a", "module_temp_c",
    "inverter_efficiency", "status_code",
]

METEO_REQUIRED = [
    "station_id", "reading_time", "ghi_wm2", "ambient_temp", "temp_unit",
]

missing_scada = [c for c in SCADA_REQUIRED if c not in scada.columns]
missing_meteo = [c for c in METEO_REQUIRED if c not in meteo.columns]

if missing_scada:
    raise KeyError(f"Kolom SCADA tidak ditemukan: {missing_scada}")
if missing_meteo:
    raise KeyError(f"Kolom meteorological tidak ditemukan: {missing_meteo}")

scada["timestamp"] = pd.to_datetime(scada["timestamp"], errors="coerce")
meteo["reading_time"] = pd.to_datetime(meteo["reading_time"], errors="coerce")

invalid_scada_timestamp = int(scada["timestamp"].isna().sum())
invalid_meteo_timestamp = int(meteo["reading_time"].isna().sum())

scada = scada.dropna(subset=["timestamp"]).copy()
meteo = meteo.dropna(subset=["reading_time"]).copy()

print("Invalid SCADA timestamps removed:", invalid_scada_timestamp)
print("Invalid meteorological timestamps removed:", invalid_meteo_timestamp)
print("SCADA time range:", scada["timestamp"].min(), "->", scada["timestamp"].max())
print("Meteo time range:", meteo["reading_time"].min(), "->", meteo["reading_time"].max())


Invalid SCADA timestamps removed: 0
Invalid meteorological timestamps removed: 0
SCADA time range: 2025-04-01 00:00:00 -> 2025-06-29 23:30:00
Meteo time range: 2025-04-01 00:00:00 -> 2025-06-29 23:45:00


## 2. Validate numeric domains and current-unit consistency

The source column is named `dc_current_a`, but the numerical relationship in this source file is consistent with current being stored on a kA scale. For example, approximately `580 V × 0.1871 = 108.5 kW`. Therefore the canonical output converts the source numeric value to amperes by multiplying by 1000.

The notebook does not remove high-power observations based on this check. They remain available for the downstream anomaly model.

In [12]:
numeric_scada = [
    "dc_power_kw", "ac_power_kw", "dc_voltage_v", "dc_current_a",
    "module_temp_c", "inverter_efficiency",
]
numeric_meteo = [
    "ghi_wm2", "dni_wm2", "dhi_wm2", "ambient_temp",
    "wind_speed_ms", "relative_humidity_pct", "barometric_pressure_hpa",
    "solar_zenith_angle_deg", "cloud_opacity_pct", "precipitation_mm",
]

for col in numeric_scada:
    scada[col] = pd.to_numeric(scada[col], errors="coerce")
for col in numeric_meteo:
    meteo[col] = pd.to_numeric(meteo[col], errors="coerce")

negative_checks = {
    "dc_power_kw": int((scada["dc_power_kw"] < 0).sum()),
    "ac_power_kw": int((scada["ac_power_kw"] < 0).sum()),
    "dc_voltage_v": int((scada["dc_voltage_v"] < 0).sum()),
    "dc_current_a": int((scada["dc_current_a"] < 0).sum()),
    "ghi_wm2": int((meteo["ghi_wm2"] < 0).sum()),
    "ambient_temp": int((meteo["ambient_temp"] < -80).sum()),
}

print("Negative/domain checks:")
print(pd.Series(negative_checks))

# Keep the original source value for auditability; canonical current is in amperes.
scada["source_dc_current_a"] = scada["dc_current_a"]
scada["dc_current_a"] = scada["dc_current_a"] * 1000.0

scada["dc_power_from_vi_kw"] = (
    scada["dc_voltage_v"] * scada["dc_current_a"] / 1000.0
)

power_denominator = scada["dc_power_from_vi_kw"].replace(0, np.nan)
scada["dc_power_consistency_ratio"] = (
    scada["dc_power_kw"] / power_denominator
)

operating = (
    scada["dc_power_kw"].notna()
    & scada["dc_power_from_vi_kw"].notna()
    & (scada["dc_power_from_vi_kw"] > 0)
)

ratio = scada.loc[operating, "dc_power_consistency_ratio"]
print("\nDC power/current physical consistency after unit normalization:")
print("Median P_DC / (V_DC*I_DC):", round(float(ratio.median()), 6))
print("P05:", round(float(ratio.quantile(0.05)), 6))
print("P95:", round(float(ratio.quantile(0.95)), 6))
print("Valid comparisons:", len(ratio))


Negative/domain checks:
dc_power_kw     0
ac_power_kw     0
dc_voltage_v    0
dc_current_a    0
ghi_wm2         0
ambient_temp    0
dtype: int64

DC power/current physical consistency after unit normalization:
Median P_DC / (V_DC*I_DC): 0.999999
P05: 0.999074
P95: 1.000973
Valid comparisons: 17915


## 3. Aggregate the three meteorological stations

Meteorological observations are available every 15 minutes from multiple stations. We first aggregate stations at each timestamp using the median, which avoids arbitrarily selecting one station. The resulting plant-level meteorological series is then resampled to 30-minute intervals to match SCADA.

In [13]:
# Inspect actual temperature units
print("Unique temp_unit values:")
print(meteo["temp_unit"].value_counts(dropna=False))

print("\nUnique values:")
print(meteo["temp_unit"].drop_duplicates().tolist())

Unique temp_unit values:
temp_unit
C    20160
F     5760
Name: count, dtype: int64

Unique values:
['C', 'F']


In [14]:
print("Weather stations:", sorted(meteo["station_id"].dropna().unique().tolist()))
print("Number of stations:", meteo["station_id"].nunique())

weather_numeric = [
    "ghi_wm2", "dni_wm2", "dhi_wm2", "ambient_temp",
    "wind_speed_ms", "relative_humidity_pct", "barometric_pressure_hpa",
    "solar_zenith_angle_deg", "cloud_opacity_pct", "precipitation_mm",
]

# Normalize obvious unit representation before aggregation.
# Normalize temperature unit representation
meteo["temp_unit"] = (
    meteo["temp_unit"]
    .astype("string")
    .str.strip()
    .str.upper()
    .str.replace("°", "", regex=False)
    .str.replace(" ", "", regex=False)
)

print("Temperature units found:")
print(meteo["temp_unit"].value_counts(dropna=False))

# Preserve original temperature before conversion
meteo["ambient_temp_source"] = meteo["ambient_temp"]

# Convert Fahrenheit to Celsius
fahrenheit_mask = meteo["temp_unit"].isin(["F", "FAHRENHEIT", "DEGF"])

meteo.loc[fahrenheit_mask, "ambient_temp"] = (
    meteo.loc[fahrenheit_mask, "ambient_temp"] - 32
) * 5 / 9

# Celsius units require no conversion
celsius_mask = meteo["temp_unit"].isin(["C", "CELSIUS", "DEGC"])

# Check unsupported units
unsupported_mask = (
    meteo["temp_unit"].notna()
    & ~meteo["temp_unit"].isin(
        ["C", "CELSIUS", "DEGC", "F", "FAHRENHEIT", "DEGF"]
    )
)

unsupported_units = meteo.loc[
    unsupported_mask, "temp_unit"
].unique()

if len(unsupported_units) > 0:
    raise ValueError(
        f"Unsupported meteorological temperature units: {unsupported_units.tolist()}"
    )

# Standardize the unit after conversion
meteo.loc[
    celsius_mask | fahrenheit_mask,
    "temp_unit"
] = "C"

print("Temperature unit after normalization:")
print(meteo["temp_unit"].value_counts(dropna=False))
meteo_plant_15m = (
    meteo.groupby("reading_time", as_index=False)[weather_numeric]
    .median()
)

station_count = (
    meteo.groupby("reading_time")["station_id"]
    .nunique()
    .rename("weather_station_count")
)

meteo_plant_15m = meteo_plant_15m.merge(
    station_count,
    left_on="reading_time",
    right_index=True,
    how="left",
)

meteo_plant_15m = meteo_plant_15m.set_index("reading_time").sort_index()

# SCADA is 30-minute. Use time-based median for numeric weather variables.
meteo_plant_30m = meteo_plant_15m[weather_numeric].resample("30min").median()

# Keep station coverage as an audit variable. For each 30-min period, use the maximum
# observed number of stations contributing to the underlying 15-min records.
meteo_plant_30m["weather_station_count"] = (
    meteo_plant_15m["weather_station_count"].resample("30min").max()
)

meteo_plant_30m = meteo_plant_30m.reset_index()

print("Aggregated 15-min rows:", len(meteo_plant_15m))
print("Aggregated 30-min rows:", len(meteo_plant_30m))
display(meteo_plant_30m.head())


Weather stations: ['WS-Central', 'WS-North', 'WS-South']
Number of stations: 3
Temperature units found:
temp_unit
C    20160
F     5760
Name: count, dtype: Int64
Temperature unit after normalization:
temp_unit
C    25920
Name: count, dtype: Int64
Aggregated 15-min rows: 8640
Aggregated 30-min rows: 4320


,reading_time,ghi_wm2,dni_wm2,dhi_wm2,ambient_temp,wind_speed_ms,relative_humidity_pct,barometric_pressure_hpa,solar_zenith_angle_deg,cloud_opacity_pct,precipitation_mm,weather_station_count
0,2025-04-01 00:00:00,4.9570,2.0480,4.9165,17.125,8.7250,58.010,1021.510,65.1990,45.310,0.2005,3
1,2025-04-01 00:30:00,6.1890,1.6455,4.4065,16.580,5.3570,51.350,1014.560,68.1110,41.605,0.1795,3
2,2025-04-01 01:00:00,12.9615,1.1100,5.1045,17.490,6.6940,60.385,1021.305,66.5125,34.440,0.2345,3
3,2025-04-01 01:30:00,6.9160,2.7850,4.2490,17.330,5.3010,59.600,1006.155,36.7210,55.875,0.2185,3
4,2025-04-01 02:00:00,4.8450,1.0610,8.9570,16.085,3.4445,60.095,1005.790,45.7225,38.190,0.3690,3


In [15]:
print("\nAmbient temperature after unit conversion:")
display(
    meteo["ambient_temp"].describe().to_frame("ambient_temp_c")
)

print(
    "Temperature < -50 C:",
    (meteo["ambient_temp"] < -50).sum()
)

print(
    "Temperature > 70 C:",
    (meteo["ambient_temp"] > 70).sum()
)


Ambient temperature after unit conversion:


,ambient_temp_c
count,25920.000000
mean,24.803211
std,5.595529
min,9.150000
25%,20.470000
50%,23.930000
75%,29.162500
max,43.266667


Temperature < -50 C: 0
Temperature > 70 C: 0


## 4. Align SCADA and plant-level meteorology

SCADA timestamps are already at 30-minute resolution. The join therefore uses the exact timestamp rather than a nearest-station lookup. Missing weather observations remain missing; they are not forward-filled or fabricated.

In [16]:
scada = scada.sort_values(["timestamp", "inverter_id"]).reset_index(drop=True)
meteo_plant_30m = meteo_plant_30m.sort_values("reading_time").reset_index(drop=True)

dataset = scada.merge(
    meteo_plant_30m,
    left_on="timestamp",
    right_on="reading_time",
    how="left",
    validate="many_to_one",
)

dataset = dataset.drop(columns=["reading_time"])

dataset = dataset.rename(columns={
    "ghi_wm2": "irradiance_w_m2",
    "ambient_temp": "ambient_temp_c",
})

final_columns = [
    "timestamp",
    "inverter_id",
    "irradiance_w_m2",
    "ambient_temp_c",
    "module_temp_c",
    "dc_power_kw",
    "ac_power_kw",
    "dc_voltage_v",
    "dc_current_a",
    "inverter_efficiency",
    "status_code",
    "weather_station_count",
    "source_dc_current_a",
    "dc_power_from_vi_kw",
    "dc_power_consistency_ratio",
]

dataset = dataset[final_columns].sort_values(
    ["timestamp", "inverter_id"]
).reset_index(drop=True)

print("Final shape:", dataset.shape)
display(dataset.head(10))


Final shape: (34560, 15)


,timestamp,inverter_id,irradiance_w_m2,ambient_temp_c,module_temp_c,dc_power_kw,ac_power_kw,dc_voltage_v,dc_current_a,inverter_efficiency,status_code,weather_station_count,source_dc_current_a,dc_power_from_vi_kw,dc_power_consistency_ratio
0,2025-04-01 00:00:00,INV-01,4.957,17.125,21.81,0.0,0.0,0.0,0.0,NaN,NORMAL,3,0.0,0.0,NaN
1,2025-04-01 00:00:00,INV-02,4.957,17.125,17.80,0.0,0.0,0.0,0.0,NaN,NORMAL,3,0.0,0.0,NaN
2,2025-04-01 00:00:00,INV-03,4.957,17.125,15.30,0.0,0.0,0.0,0.0,NaN,NORMAL,3,0.0,0.0,NaN
3,2025-04-01 00:00:00,INV-04,4.957,17.125,23.44,0.0,0.0,0.0,0.0,NaN,NORMAL,3,0.0,0.0,NaN
4,2025-04-01 00:00:00,INV-05,4.957,17.125,16.70,0.0,0.0,0.0,0.0,NaN,NORMAL,3,0.0,0.0,NaN
5,2025-04-01 00:00:00,INV-06,4.957,17.125,17.23,0.0,0.0,0.0,0.0,NaN,NORMAL,3,0.0,0.0,NaN
6,2025-04-01 00:00:00,INV-07,4.957,17.125,18.21,0.0,0.0,0.0,0.0,NaN,NORMAL,3,0.0,0.0,NaN
7,2025-04-01 00:00:00,INV-08,4.957,17.125,24.49,0.0,0.0,0.0,0.0,NaN,NORMAL,3,0.0,0.0,NaN
8,2025-04-01 00:30:00,INV-01,6.189,16.580,23.11,0.0,0.0,0.0,0.0,NaN,NORMAL,3,0.0,0.0,NaN
9,2025-04-01 00:30:00,INV-02,6.189,16.580,26.09,0.0,0.0,0.0,0.0,NaN,NORMAL,3,0.0,0.0,NaN


In [19]:
# Irradiance distribution
print("Irradiance statistics:")
display(meteo["ghi_wm2"].describe().to_frame("ghi_wm2"))

# Low irradiance observations
print("\nIrradiance <= 50 W/m²:")
print((meteo["ghi_wm2"] <= 50).sum())

print("\nIrradiance 50–200 W/m²:")
print(
    (
        (meteo["ghi_wm2"] > 50) &
        (meteo["ghi_wm2"] < 200)
    ).sum()
)

print("\nIrradiance >= 200 W/m²:")
print((meteo["ghi_wm2"] >= 200).sum())

Irradiance statistics:


,ghi_wm2
count,25920.000000
mean,290.099870
std,336.940904
min,0.000000
25%,5.758750
50%,93.006000
75%,571.864250
max,1230.042000



Irradiance <= 50 W/m²:
12446

Irradiance 50–200 W/m²:
1765

Irradiance >= 200 W/m²:
11709


In [20]:
meteo_check = meteo.copy()
meteo_check["hour"] = meteo_check["reading_time"].dt.hour

hourly_irradiance = (
    meteo_check
    .groupby("hour")["ghi_wm2"]
    .agg(["count", "mean", "min", "max"])
)

display(hourly_irradiance)

,count,mean,min,max
hour,,,,
0,1080,6.163244,0.010,27.114
1,1080,6.262052,0.006,30.058
2,1080,6.345806,0.014,25.317
3,1080,6.222581,0.007,25.266
4,1080,6.216256,0.005,29.944
5,1080,17.339082,0.000,95.391
6,1080,175.546267,28.108,378.152
7,1080,368.216231,92.723,636.898
8,1080,534.469283,90.271,820.204


## 5. Final validation

The checks below validate schema, uniqueness, missingness, physical consistency, and preservation of the original DC/AC SCADA variables. No anomaly/fault label is generated.

In [17]:
print("=== DATASET VALIDATION ===")
print("Rows:", f"{len(dataset):,}")
print("Inverters:", dataset["inverter_id"].nunique())
print("Unique timestamps:", dataset["timestamp"].nunique())
print("Duplicate inverter-timestamp:", dataset.duplicated(["inverter_id", "timestamp"]).sum())

print("\nMissing values:")
display(dataset.isna().sum().to_frame("missing"))

print("\nCurrent unit conversion:")
print("Source column: dc_current_a")
print("Output column: dc_current_a")
print("Conversion: source value × 1000")

valid_ratio = dataset["dc_power_consistency_ratio"].replace([np.inf, -np.inf], np.nan).dropna()
print("\nDC physical consistency ratio summary:")
display(valid_ratio.describe().to_frame("dc_power_consistency_ratio"))

print("\nAC/DC efficiency summary from source column:")
display(dataset["inverter_efficiency"].describe().to_frame("inverter_efficiency"))

print("\nWeather station coverage:")
display(dataset["weather_station_count"].value_counts(dropna=False).sort_index().to_frame("rows"))

print("\nPreserved SCADA variables:")
print([
    "dc_power_kw", "ac_power_kw", "dc_voltage_v", "dc_current_a",
    "module_temp_c", "inverter_efficiency", "status_code",
])


=== DATASET VALIDATION ===
Rows: 34,560
Inverters: 8
Unique timestamps: 4320
Duplicate inverter-timestamp: 0

Missing values:


,missing
timestamp,0
inverter_id,0
irradiance_w_m2,0
ambient_temp_c,0
module_temp_c,0
dc_power_kw,180
ac_power_kw,180
dc_voltage_v,0
dc_current_a,0
inverter_efficiency,16567



Current unit conversion:
Source column: dc_current_a
Output column: dc_current_a
Conversion: source value × 1000

DC physical consistency ratio summary:


,dc_power_consistency_ratio
count,17915.000000
mean,20.110088
std,136.904428
min,0.000000
25%,0.999781
50%,0.999999
75%,1.000218
max,1002.605250



AC/DC efficiency summary from source column:


,inverter_efficiency
count,17993.000000
mean,0.969957
std,0.008663
min,0.955000
25%,0.962460
50%,0.970020
75%,0.977470
max,0.985000



Weather station coverage:


,rows
weather_station_count,
3,34560



Preserved SCADA variables:
['dc_power_kw', 'ac_power_kw', 'dc_voltage_v', 'dc_current_a', 'module_temp_c', 'inverter_efficiency', 'status_code']


In [18]:
dataset.to_csv(OUTPUT_FILE, index=False)

print(f"Validated dataset saved: {OUTPUT_FILE}")
print(f"Rows: {len(dataset):,}")
print(f"Columns: {dataset.columns.tolist()}")


Validated dataset saved: output\pv_monitoring_dataset_validated.csv
Rows: 34,560
Columns: ['timestamp', 'inverter_id', 'irradiance_w_m2', 'ambient_temp_c', 'module_temp_c', 'dc_power_kw', 'ac_power_kw', 'dc_voltage_v', 'dc_current_a', 'inverter_efficiency', 'status_code', 'weather_station_count', 'source_dc_current_a', 'dc_power_from_vi_kw', 'dc_power_consistency_ratio']


## Methodological notes

1. The SCADA variables retain their original DC/AC semantics instead of collapsing them into generic `power_kw`, `voltage_v`, and `current_a` columns.
2. The numeric source `dc_current_a` is converted to canonical amperes because its observed scale is consistent with kA when checked against `dc_power_kw = dc_voltage_v × dc_current / 1000`.
3. `source_dc_current_a` is retained so the unit transformation remains auditable.
4. Three meteorological stations are aggregated by timestamp using the median before resampling to 30-minute plant-level data.
5. Weather alignment uses exact 30-minute timestamps and does not select one station arbitrarily.
6. Missing SCADA measurements are preserved. No interpolation or anomaly label is generated in this notebook.
7. `dc_power_from_vi_kw` and `dc_power_consistency_ratio` are diagnostic physical-consistency features; they are not fault labels.
8. Extreme values present in the source SCADA are retained so the downstream anomaly model can evaluate them rather than having the dataset builder remove them.
